In [5]:
import csv
import re
import time
from typing import Optional, Tuple
from pathlib import Path
from os.path import expanduser

import lxml.html as lh
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

_driver = None

LOCATION_XPATH = '//*[@id="inner-content"]/div[1]/app-dashboard-header/div[2]/div/div[1]/span'
FALLBACK_LOCATION_XPATH = '//app-dashboard-header//span[1]'
ROOT = Path(expanduser('~/phenology-urban/data/raw/CONUS/WU'))

In [6]:
def is_js_placeholder(text: str) -> bool:
    t = (text or '')[:4000]
    return any(tok.lower() in t.lower() for tok in BLOCK_TOKENS)

def get_driver():
    global _driver
    if _driver is not None:
        return _driver
    opts = uc.ChromeOptions()
    opts.add_argument("--headless=new")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--window-size=1200,900")
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/124.0.0.0 Safari/537.36")
    _driver = uc.Chrome(options=opts)
    return _driver

def make_session():
    s = requests.Session()
    retries = Retry(total=3, backoff_factor=0.5,
                    status_forcelist=(429, 500, 502, 503, 504))
    adapter = HTTPAdapter(max_retries=retries,
                          pool_connections=GLOBAL_MAX_CONNECTIONS,
                          pool_maxsize=GLOBAL_MAX_CONNECTIONS)
    s.mount('http://', adapter)
    s.mount('https://', adapter)
    s.headers.update({
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/124.0.0.0 Safari/537.36"),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive",
        "Referer": "https://www.wunderground.com/"
    })
    return s

_LOC_RE = re.compile(r"""
    Elev(?:ation)?\s+
    (?P<elev_val>[-+]?\d+(?:\.\d+)?)      # elevation number
    \s*
    (?P<elev_unit>m|ft)                   # unit
    .*?                                   # anything (comma, spaces)
    (?P<lat_val>[-+]?\d+(?:\.\d+)?)       # latitude value
    \s*°?\s*
    (?P<lat_hem>[NS])                     # N/S
    \s*,\s*
    (?P<lon_val>[-+]?\d+(?:\.\d+)?)       # longitude value
    \s*°?\s*
    (?P<lon_hem>[EW])                     # E/W
""", re.IGNORECASE | re.VERBOSE)

def _to_signed(value: float, hemi: str) -> float:
    hemi = hemi.upper()
    if hemi in ("S", "W"):
        return -abs(value)
    return abs(value)

def parse_location_text(text: str) -> Tuple[Optional[float], Optional[float], Optional[float]]:
    """
    Returns (ele_m, lon_dd, lat_dd) or (None, None, None) if not parseable.
    - ele_m: elevation in meters (float)
    - lon_dd: longitude in decimal degrees (E positive, W negative)
    - lat_dd: latitude in decimal degrees (N positive, S negative)
    """
    if not text:
        return (None, None, None)

    m = _LOC_RE.search(text)
    if m:
        elev_val = float(m.group("elev_val"))
        elev_unit = m.group("elev_unit").lower()
        ele_m = elev_val if elev_unit == "m" else elev_val * 0.3048

        lat = _to_signed(float(m.group("lat_val")), m.group("lat_hem"))
        lon = _to_signed(float(m.group("lon_val")), m.group("lon_hem"))
        return (round(ele_m, 3), round(lon, 6), round(lat, 6))

    return (None, None, None)

def to_station_url(station: str) -> str:
    return f"https://www.wunderground.com/dashboard/pws/{station}"

def extract_location_from_html(content: bytes) -> Optional[str]:
    doc = lh.fromstring(content)
    els = doc.xpath(LOCATION_XPATH) or doc.xpath(FALLBACK_LOCATION_XPATH)
    if not els:
        return None
    return ' '.join(els[0].text_content().split())

_driver = None

def get_driver():
    global _driver
    if _driver is not None:
        return _driver
    opts = uc.ChromeOptions()
    opts.add_argument("--headless=new")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--window-size=1200,900")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    # 减少自动化痕迹（可选）
    opts.add_argument("--disable-blink-features=AutomationControlled")
    _driver = uc.Chrome(options=opts)
    return _driver

def maybe_accept_consent(drv, timeout=3):
    """尝试点掉 GDPR/隐私弹窗（如果存在则点击，不存在就忽略）"""
    try:
        WebDriverWait(drv, timeout).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        ).click()
        time.sleep(0.2)
    except Exception:
        pass

def extract_location_text_with_driver(drv, timeout: int = 15) -> str:
    """优先严格 XPath，失败再 fallback，返回清洗后的文本（可能为空字符串）"""
    try:
        WebDriverWait(drv, timeout).until(
            EC.presence_of_element_located((By.XPATH, LOCATION_XPATH))
        )
        text = drv.find_element(By.XPATH, LOCATION_XPATH).text.strip()
        if text:
            return ' '.join(text.split())
    except Exception:
        pass

    try:
        WebDriverWait(drv, max(5, timeout // 2)).until(
            EC.presence_of_element_located((By.XPATH, FALLBACK_LOCATION_XPATH))
        )
        text = drv.find_element(By.XPATH, FALLBACK_LOCATION_XPATH).text.strip()
        if text:
            return ' '.join(text.split())
    except Exception:
        pass

    # 最后兜底，用整个页面源码抓到的文本再尝试解析
    try:
        doc = lh.fromstring(drv.page_source.encode("utf-8", errors="ignore"))
        els = doc.xpath(LOCATION_XPATH) or doc.xpath(FALLBACK_LOCATION_XPATH)
        if els:
            return ' '.join(els[0].text_content().split())
    except Exception:
        pass
    return ""

# ========= 目录与城市-站点 =========
def get_city_to_stations(root: Path) -> dict[str, list[str]]:
    """按城市文件夹分组 station（文件名 stem），并去重排序"""
    city_map: dict[str, set[str]] = {}
    for p in root.glob('*/*.csv'):
        city = p.parent.name
        station = p.stem
        city_map.setdefault(city, set()).add(station)
    return {city: sorted(sts) for city, sts in city_map.items()}

# ========= 核心：仅用浏览器获取位置 =========
def fetch_station_location_browser_only(drv, station: str, timeout: int = 15) -> dict:
    url = to_station_url(station)
    try:
        drv.get(url)
        maybe_accept_consent(drv)
        text = extract_location_text_with_driver(drv, timeout=timeout)
        ele, lon, lat = parse_location_text(text or "")
        return {"station": station, "url": url, "ele": ele, "lon": lon, "lat": lat}
    except Exception:
        return {"station": station, "url": url, "ele": None, "lon": None, "lat": None}

def build_locations_for_city(city: str, stations: list[str]) -> list[dict]:
    """同一个城市复用一个浏览器实例，顺序抓取（更稳、更少被封）"""
    drv = get_driver()
    rows = []
    for s in stations:
        rows.append(fetch_station_location_browser_only(drv, s, timeout=15))
        time.sleep(0.2)  # 轻微节流，降低触发风控概率
    return rows

def write_city_locations(city: str, rows: list[dict], root: Path):
    out_path = root / city / 'location.csv'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['station', 'url', 'ele', 'lon', 'lat'])
        writer.writeheader()
        writer.writerows(rows)
    print(f"[{city}] wrote {out_path} ({len(rows)} stations)")

In [7]:
def main():
    city_to_stations = get_city_to_stations(ROOT)
    if not city_to_stations:
        print(f"No stations found under {ROOT}")
        return

    for city, stations in sorted(city_to_stations.items()):
        print(f"=== {city}: {len(stations)} stations ===")
        rows = build_locations_for_city(city, stations)
        write_city_locations(city, rows, ROOT)

    # 退出浏览器
    try:
        drv = get_driver()
        drv.quit()
    except Exception:
        pass

if __name__ == "__main__":
    main()

=== Albuquerque: 756 stations (test) ===
[Albuquerque] wrote /home/jializ/phenology-urban/data/raw/CONUS/WU/Albuquerque/location.csv (756 stations)
